# Robustesse de la détection d'objets face à la météo — YOLOv8 et RT-DETR

Images de caméras de contrôle routier (5 classes : plaque, téléphone, véhicule, ceinture, pare-brise),
dégradées par 5 conditions météo (brouillard, nuit, pluie, neige, éblouissement) à 5 niveaux.

**Protocole :** on entraîne le modèle sur les images **originales** uniquement, puis on le teste sur les
**mêmes images de test** dans chacune des 26 conditions. La baisse de mAP mesure la robustesse.

Avant de commencer : *Exécution > Modifier le type d'exécution > GPU T4*.

In [ ]:
!pip install -q ultralytics
import os, random, glob, zipfile
import pandas as pd
from ultralytics import YOLO

## 1. Récupérer le dataset depuis Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP = '/content/drive/MyDrive/results.zip'   # le zip du dossier results, posé à la racine de ton Drive
if not os.path.exists('/content/results'):
    with zipfile.ZipFile(ZIP) as z:
        z.extractall('/content')
DATA = '/content/results'
CONDITIONS = sorted(os.listdir(DATA))

# YOLO cherche les labels dans un dossier « labels » à côté de « images »
for c in CONDITIONS:
    if os.path.exists(f'{DATA}/{c}/labels_yolo') and not os.path.exists(f'{DATA}/{c}/labels'):
        os.rename(f'{DATA}/{c}/labels_yolo', f'{DATA}/{c}/labels')
print(len(CONDITIONS), 'conditions :', CONDITIONS)


## 2. Découpage train / val / test (70 / 15 / 15, graine 42)
Le découpage est fait une seule fois sur les images originales, puis réutilisé pour toutes les conditions.

In [ ]:
CLASSES = ['license_plate', 'phone', 'vehicle', 'seatbelt', 'windshield']
random.seed(42)
noms = sorted(os.listdir(f'{DATA}/original/images'))
random.shuffle(noms)
n = len(noms)
split = {'train': noms[:int(.7*n)], 'val': noms[int(.7*n):int(.85*n)], 'test': noms[int(.85*n):]}
print({k: len(v) for k, v in split.items()})

os.makedirs('/content/splits', exist_ok=True)
def ecrire_liste(condition, partie):
    chemin = f'/content/splits/{condition}_{partie}.txt'
    with open(chemin, 'w') as f:
        for nom in split[partie]:
            img = f'{DATA}/{condition}/images/{nom}'
            if os.path.exists(img):
                f.write(img + '\n')
    return chemin

def ecrire_yaml(nom, train, val):
    chemin = f'/content/splits/{nom}.yaml'
    with open(chemin, 'w') as f:
        f.write(f'train: {train}\nval: {val}\nnc: {len(CLASSES)}\nnames: {CLASSES}\n')
    return chemin

yaml_entrainement = ecrire_yaml('original', ecrire_liste('original', 'train'), ecrire_liste('original', 'val'))

## 3. Entraîner YOLOv8 sur les images originales

In [ ]:
modele = YOLO('yolov8s.pt')
modele.train(data=yaml_entrainement, epochs=50, imgsz=640, batch=16, seed=42,
             project='/content/drive/MyDrive/robustesse_meteo', name='yolov8s', exist_ok=True)

## 4. Tester le modèle sur les 26 conditions

In [ ]:
meilleur = YOLO('/content/drive/MyDrive/robustesse_meteo/yolov8s/weights/best.pt')
resultats = []
for c in CONDITIONS:
    y = ecrire_yaml(f'test_{c}', ecrire_liste('original', 'train'), ecrire_liste(c, 'test'))
    m = meilleur.val(data=y, imgsz=640, batch=16, verbose=False, plots=False)
    meteo, niveau = ('original', 0) if c == 'original' else (c.split('_level')[0], int(c.split('_level')[1]))
    resultats.append({'modele': 'YOLOv8s', 'meteo': meteo, 'niveau': niveau,
                      'mAP50': m.box.map50, 'mAP50_95': m.box.map})
    print(c, round(m.box.map50, 3), round(m.box.map, 3))

df = pd.DataFrame(resultats)
df.to_csv('/content/drive/MyDrive/robustesse_meteo/resultats_yolov8s.csv', index=False)
df

## 5. Visualiser la baisse de performance

In [ ]:
import matplotlib.pyplot as plt
ref = df[df.meteo == 'original'].mAP50.iloc[0]
plt.figure(figsize=(8, 5))
for meteo, g in df[df.meteo != 'original'].groupby('meteo'):
    g = g.sort_values('niveau')
    plt.plot([0] + list(g.niveau), [ref] + list(g.mAP50), marker='o', label=meteo)
plt.xlabel('Niveau de dégradation (0 = image originale)')
plt.ylabel('mAP@50')
plt.title('YOLOv8s : robustesse face à la météo')
plt.legend(); plt.grid(alpha=.3)
plt.savefig('/content/drive/MyDrive/robustesse_meteo/courbe_yolov8s.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Deuxième modèle : RT-DETR (transformer)
Même protocole : entraînement sur les originales, test sur les 26 conditions.

In [ ]:
from ultralytics import RTDETR

rtdetr = RTDETR('rtdetr-l.pt')
rtdetr.train(data=yaml_entrainement, epochs=50, imgsz=640, batch=8, seed=42,
             project='/content/drive/MyDrive/robustesse_meteo', name='rtdetr-l', exist_ok=True)

meilleur_rtdetr = RTDETR('/content/drive/MyDrive/robustesse_meteo/rtdetr-l/weights/best.pt')
resultats_rtdetr = []
for c in CONDITIONS:
    y = ecrire_yaml(f'test_{c}', ecrire_liste('original', 'train'), ecrire_liste(c, 'test'))
    m = meilleur_rtdetr.val(data=y, imgsz=640, batch=8, verbose=False, plots=False)
    meteo, niveau = ('original', 0) if c == 'original' else (c.split('_level')[0], int(c.split('_level')[1]))
    resultats_rtdetr.append({'modele': 'RT-DETR-l', 'meteo': meteo, 'niveau': niveau,
                             'mAP50': m.box.map50, 'mAP50_95': m.box.map})
    print(c, round(m.box.map50, 3), round(m.box.map, 3))

df_rtdetr = pd.DataFrame(resultats_rtdetr)
df_rtdetr.to_csv('/content/drive/MyDrive/robustesse_meteo/resultats_rtdetr-l.csv', index=False)
df_rtdetr

## 7. Comparer les deux modèles

In [ ]:
tous = pd.concat([pd.read_csv('/content/drive/MyDrive/robustesse_meteo/resultats_yolov8s.csv'), df_rtdetr])
fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=True)
for ax, meteo in zip(axes, ['fog', 'night', 'rain', 'snow', 'glare']):
    for modele, g in tous.groupby('modele'):
        ref = g[g.meteo == 'original'].mAP50.iloc[0]
        g = g[g.meteo == meteo].sort_values('niveau')
        ax.plot([0] + list(g.niveau), [ref] + list(g.mAP50), marker='o', label=modele)
    ax.set_title(meteo); ax.set_xlabel('Niveau'); ax.grid(alpha=.3)
axes[0].set_ylabel('mAP@50'); axes[0].legend()
plt.suptitle('YOLOv8s contre RT-DETR-l face à la météo')
plt.savefig('/content/drive/MyDrive/robustesse_meteo/comparaison_modeles.png', dpi=150, bbox_inches='tight')
plt.show()